# BT-FC-DO aplicado al problema de coloreado de grafos

### Problemas de Satisfacción de Restricciones (CSP)

**Algoritmo:** Backtracking + Forward Checking + Dynamic Ordering

El objetivo es implementar un algoritmo BT-FC-DO para resolver el problema
de coloreado de grafos.

El grafo se representa mediante una matriz de adyacencia simétrica de
tamaño N × N, donde:

- 1 representa que dos vértices están conectados.
- 0 representa que no existe conexión.
- N < 21.

El algoritmo debe encontrar una coloración válida y determinar el
número cromático mínimo del grafo.

### Librerias

In [3]:
try:
    import random
    import matplotlib.pyplot as plt
    import numpy as np
    import networkx as nx
except ImportError:
    %pip install numpy matplotlib networkx
    import random
    import matplotlib.pyplot as plt
    import numpy as np
    import networkx as nx


## Generación del grafo

Para realizar las pruebas podemos generar un grafo aleatoriamente.

La matriz de adyacencia debe ser simétrica porque el grafo es no dirigido.

Por lo tanto:

A[i][j] = A[j][i]

Además, no se permiten conexiones de un vértice consigo mismo:

A[i][i] = 0

In [2]:

def generar_grafo(n, probabilidad=0.35):
    """
    Genera una matriz de adyacencia simétrica.

    n: número de vértices
    probabilidad: probabilidad de que exista una arista
    """

    if n <= 0 or n >= 21:
        raise ValueError("N debe cumplir 0 < N < 21")

    matriz = np.zeros((n, n), dtype=int)

    for i in range(n):
        for j in range(i + 1, n):

            if random.random() < probabilidad:
                matriz[i][j] = 1
                matriz[j][i] = 1

    return matriz

N = 8

matriz = generar_grafo(
    N,
    probabilidad=0.35
)

print("Matriz de adyacencia:")
print(matriz)

NameError: name 'np' is not defined

## Lectura desde archivo

La matriz también puede ser cargada desde un archivo de texto.

Cada fila del archivo representa una fila de la matriz de adyacencia.

In [ ]:
def leer_matriz_txt(nombre_archivo):

    matriz = []

    with open(nombre_archivo, "r") as archivo:

        for linea in archivo:

            linea = linea.strip()

            if linea:
                fila = list(map(int, linea.split()))
                matriz.append(fila)

    matriz = np.array(matriz)

    return matriz

matriz = leer_matriz_txt("ejemplo.txt")

Validar matriz

In [ ]:
def validar_matriz(matriz):

    n = len(matriz)

    # Verificar tamaño
    if n == 0 or n >= 21:
        return False

    # Verificar que sea cuadrada
    if matriz.shape != (n, n):
        return False

    # Verificar valores
    if not np.all(np.isin(matriz, [0, 1])):
        return False

    # Verificar simetría
    if not np.array_equal(matriz, matriz.T):
        return False

    # No debe haber lazos
    if np.any(np.diag(matriz) != 0):
        return False

    return True

if validar_matriz(matriz):
    print("Matriz válida.")
else:
    print("Matriz inválida.")

## Representación gráfica

Aunque el algoritmo trabaja directamente con la matriz de adyacencia,
se puede visualizar el grafo para facilitar la interpretación del problema.

In [ ]:
def visualizar_grafo(matriz, colores=None):

    G = nx.Graph()

    n = len(matriz)

    for i in range(n):
        G.add_node(i)

    for i in range(n):
        for j in range(i + 1, n):

            if matriz[i][j] == 1:
                G.add_edge(i, j)

    posiciones = nx.spring_layout(G, seed=10)

    plt.figure(figsize=(8, 6))

    if colores is None:
        nx.draw(
            G,
            posiciones,
            with_labels=True,
            node_size=800
        )
    else:
        nx.draw(
            G,
            posiciones,
            with_labels=True,
            node_color=colores,
            node_size=800
        )

    plt.title("Grafo")
    plt.show()

# Formulación del problema como CSP

El problema de coloreado se representa como un CSP.

### Variables

Cada vértice del grafo representa una variable.

Por ejemplo, para un grafo de 5 vértices:

X = {0, 1, 2, 3, 4}

### Dominios

Cada vértice puede recibir uno de los colores disponibles.

Para k = 3:

D0 = {A, B, C}
D1 = {A, B, C}
D2 = {A, B, C}
...

### Restricciones

Si dos vértices están conectados:

A[i][j] = 1

entonces:

color(i) != color(j)